# CineData — Validação da Gold e Desafio de Analytics

Este notebook confere as nove tabelas Gold e resolve as seis perguntas do enunciado.
Execute depois dos notebooks Bronze, Silver e Gold, de cima para baixo.
O arquivo foi preparado sem execução no Databricks; as saídas serão produzidas ao executar as células.
Ajuste o nome do catálogo na primeira célula para o mesmo usado no seu projeto.

As consultas financeiras e de popularidade utilizam a fato, cujo recorte é de filmes com status
**Lançado**, excluindo datas futuras. Filmes sem data válida permanecem na fato se o status for
Lançado, mas não entram nos recortes temporais das perguntas 5 e 6.

In [0]:
catalog = "cinedata_medallion"

bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

In [0]:
from datetime import datetime
from pyspark.sql import Row, functions as F

# Use o mesmo catálogo dos notebooks anteriores.
catalog = "cinedata_medallion"
gold_schema = f"{catalog}.gold"

df_movies = spark.table(f"{gold_schema}.dim_movies")
df_fact = spark.table(f"{gold_schema}.fact_movies_performance")
df_genres = spark.table(f"{gold_schema}.dim_genres")
df_people = spark.table(f"{gold_schema}.dim_people")
df_companies = spark.table(f"{gold_schema}.dim_companies")
df_reviews = spark.table(f"{gold_schema}.dim_reviews")
df_bridge_genre = spark.table(f"{gold_schema}.bridge_movie_genre")
df_bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
df_bridge_company = spark.table(f"{gold_schema}.bridge_movie_company")


## 1. Receita total em reais

Soma a receita conhecida de todos os filmes presentes na fato Gold, que contém o recorte de lançados.
SUM ignora valores NULL. Ausência de receita não é convertida em zero nem estimada.

In [0]:
df_receita_total = df_fact.agg(
    F.sum("receita_brl").alias("receita_total_brl")
)

display(df_receita_total)


receita_total_brl
808186091331.71


## 2. Cinco filmes com maior popularidade

Exclui popularidade NULL, ordena de forma decrescente e mostra cinco filmes.
A chave do filme desempata popularidades iguais para determinar as cinco linhas exibidas.

In [0]:
df_mais_populares = (
    df_fact
    .join(df_movies.select("sk_movie_id", "titulo"), on="sk_movie_id", how="inner")
    .filter(F.col("popularidade").isNotNull())
    .orderBy(F.desc("popularidade"), F.asc("sk_movie_id"))
    .select("titulo", "popularidade")
    .limit(5)
)

display(df_mais_populares)


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The I and S of Lives,2020.0
The Fear Footage 2: Curse of the Tape,2019.0


## 3. Quantidade de filmes por gênero

Usa a ponte de gêneros e conta filmes distintos, incluindo todos os filmes associados à dim_movies.
Um filme com vários gêneros participa da contagem de cada gênero. Portanto, a soma entre gêneros
não precisa ser igual à quantidade de filmes. O LEFT JOIN também permite mostrar gêneros com zero filmes.

In [0]:
df_filmes_por_genero = (
    df_genres.alias("g")
    .join(df_bridge_genre.alias("b"), on="sk_genre_id", how="left")
    .groupBy("g.nome_genero")
    .agg(F.countDistinct("b.sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.desc("qtd_filmes"), F.asc("nome_genero"))
)

display(df_filmes_por_genero)


nome_genero,qtd_filmes
Drama,32287
Documentary,18996
Comedy,18625
Thriller,10276
Horror,9729
Romance,7639
Action,6049
Crime,4747
Animation,4470
TV Movie,4080


## 4. Dez filmes de maior receita e posição com RANK()

RANK() é exigido pelo enunciado e aqui é aplicado em SQL, sem importar a API Window do Python.
A cláusula OVER é necessária para RANK(). Receitas iguais compartilham a mesma posição, com lacunas
nas posições seguintes. LIMIT 10 apresenta exatamente até dez filmes; a chave desempata quais
filmes aparecem se um empate atingir a décima linha. O ranking é calculado por receita em USD.

In [0]:
df_ranking_receita = spark.sql(f"""
    WITH ranking AS (
        SELECT
            f.sk_movie_id,
            m.titulo,
            f.receita_usd,
            f.receita_brl,
            f.popularidade,
            RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
        FROM {gold_schema}.fact_movies_performance AS f
        INNER JOIN {gold_schema}.dim_movies AS m
            ON f.sk_movie_id = m.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    SELECT titulo, receita_usd, receita_brl, posicao_ranking, popularidade
    FROM ranking
    ORDER BY posicao_ranking ASC, sk_movie_id ASC
    LIMIT 10
""")

display(df_ranking_receita)


titulo,receita_usd,receita_brl,posicao_ranking,popularidade
Avengers: Endgame,2800000000.00,13980120000.00,1,91.756
Avatar: The Way of Water,2320250281.00,11584777628.00,2,241.285
AVENGERS: INFINITY WAR,2052415039.00,10247503048.22,3,154.34
spider-man: no way home,1921847111.00,9595590440.51,4,186.065
The Lion King,1663075401.00,8303569169.65,5,63.351
Top Gun: Maverick,1488732821.00,7433094101.97,6,126.291
Barbie,1428545028.00,7132582470.30,7,1069.34
The Super Mario Bros. Movie,1355725263.00,6769000665.63,8,410.411
Black Panther,1349926083.00,6740045939.81,9,43.665
Star Wars: The Last Jedi,1332698830.00,6654031988.31,10,47.241


## Data-base das perguntas 5 e 6

A data final é a maior data de lançamento efetivamente realizada na base: status Lançado,
data preenchida e não posterior ao dia da execução. O limite NÃO é a data atual.
Os limites inferiores são calculados subtraindo 24 e 60 meses, com intervalos inclusivos.

In [0]:
df_filmes_lancados = df_movies.filter(
    (F.col("status_filme") == "Lançado")
    & F.col("data_lancamento").isNotNull()
    & (F.col("data_lancamento") <= F.current_date())
)

data_base = df_filmes_lancados.agg(
    F.max("data_lancamento").alias("data_base")
).first()["data_base"]

if data_base is None:
    raise ValueError("Não há lançamento realizado com data válida para definir os períodos.")

df_periodos = spark.range(1).select(
    F.lit(data_base).alias("data_limite_superior"),
    F.add_months(F.lit(data_base), -24).alias("inicio_ultimos_2_anos"),
    F.add_months(F.lit(data_base), -60).alias("inicio_ultimos_5_anos")
)

display(df_periodos)


data_limite_superior,inicio_ultimos_2_anos,inicio_ultimos_5_anos
2026-02-19,2024-02-19,2021-02-19


## 5. Ator com mais participações nos últimos dois anos

Conta filmes distintos por ator dentro do período, sem contar duplicatas de associações.
Em caso de empate, mostra todos os atores com a maior quantidade de participações.

In [0]:
df_participacoes_atores = (
    df_filmes_lancados
    .filter(F.col("data_lancamento").between(
        F.add_months(F.lit(data_base), -24), F.lit(data_base)
    ))
    .select("sk_movie_id")
    .join(df_bridge_person, on="sk_movie_id", how="inner")
    .join(df_people, on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_person_id", "nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
)

maior_quantidade = df_participacoes_atores.agg(
    F.max("qtd_filmes").alias("maior_quantidade")
).first()["maior_quantidade"]

df_ator_lider = (
    df_participacoes_atores
    .filter(F.col("qtd_filmes") == F.lit(maior_quantidade))
    .select("nome_pessoa", "qtd_filmes")
    .orderBy("nome_pessoa")
)

display(df_ator_lider)


nome_pessoa,qtd_filmes
Kevin Hart,64


## 6. Produtora com maior lucro nos últimos cinco anos

Soma o lucro em reais dos filmes do período ligados a cada produtora.
Como a base não informa percentuais de participação, o lucro integral de cada filme é atribuído
a cada produtora associada. Não some esses totais para obter o lucro geral da base: coproduções
aparecerão em mais de uma produtora. Lucro NULL não é inventado como zero; prejuízos são preservados.
Em caso de empate, mostra todas as produtoras líderes.

In [0]:
df_lucro_produtoras = (
    df_filmes_lancados
    .filter(F.col("data_lancamento").between(
        F.add_months(F.lit(data_base), -60), F.lit(data_base)
    ))
    .select("sk_movie_id")
    .join(df_fact.select("sk_movie_id", "lucro_brl"), on="sk_movie_id", how="inner")
    .join(df_bridge_company, on="sk_movie_id", how="inner")
    .join(df_companies, on="sk_company_id", how="inner")
    .filter(F.col("lucro_brl").isNotNull())
    .groupBy("sk_company_id", "nome_produtora")
    .agg(F.sum("lucro_brl").alias("lucro_total_brl"))
)

maior_lucro = df_lucro_produtoras.agg(
    F.max("lucro_total_brl").alias("maior_lucro")
).first()["maior_lucro"]

df_produtora_lider = (
    df_lucro_produtoras
    .filter(F.col("lucro_total_brl") == F.lit(maior_lucro))
    .select("nome_produtora", "lucro_total_brl")
    .orderBy("nome_produtora")
)

display(df_produtora_lider)


nome_produtora,lucro_total_brl
Universal Pictures,28820664854.27
